## Step 1: Environment Setup
Mount Google Drive and import necessary libraries.

In [ ]:
#CELL 1
#Mounting a new folder from google colab onto drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import rasterio
import numpy as np
import ee
from google.colab import auth
import math
import requests
from urllib.parse import urlparse, unquote
import os
import logging

# Configure logging for better progress tracking
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logging.info("Loading merged_with_terrain.csv from Google Drive...")
csv_path = '/content/drive/MyDrive/Shared_data/merged_with_terrain.csv'
df_merged = pd.read_csv(csv_path)
logging.info(f"Successfully loaded {len(df_merged)} rows from {csv_path}")
display(df_merged.head())

,pyroCb_id,time,pixel_latitude,pixel_longitude,latitude_vegetation,longitude_vegetation,cvh,cvl,tvh,tvl,...,cin,slhf,sshf,fg10,wind_speed10,wind_dir_deg,cin_filled,injection_potential,PII,capped_flag
0,179,5/27/2021 6:00,33.287442,-108.588483,33.25,-108.5,0.888824,0.093579,3,16,...,NaN,-9385.0,30882.0,4.455146,2.046433,118.403705,0.0,-0.214816,-0.019456,False
1,179,5/27/2021 12:00,33.287442,-108.588483,33.25,-108.5,0.888824,0.093579,3,16,...,NaN,-11440.0,53303.0,3.693646,1.874933,199.876023,0.0,-0.405668,-0.215182,False
2,179,5/27/2021 18:00,33.287442,-108.588483,33.25,-108.5,0.888824,0.093579,3,16,...,NaN,-257449.0,-1756259.0,11.259994,3.837899,49.726855,0.0,-0.183975,-0.223115,False
3,179,5/28/2021 0:00,33.287442,-108.588483,33.25,-108.5,0.888824,0.093579,3,16,...,NaN,-181576.0,-773753.0,13.473206,4.348494,56.359226,0.0,-0.199941,-0.204530,False
4,179,5/28/2021 6:00,33.287442,-108.588483,33.25,-108.5,0.888824,0.093579,3,16,...,NaN,-7218.0,30642.0,4.098062,2.365244,149.384452,0.0,-0.449490,-0.247899,False


## Step 2: Data Loading
Load the base CSV file containing fire hotspot locations.

### Preparing for Google Earth Engine Processing

This step involves authenticating with Google Earth Engine and identifying unique events to process. For each unique `pyroCb_id`, we will fetch terrain data once and apply it to all corresponding rows.

## Step 3: Google Earth Engine Authentication
Sign in to GEE and prepare unique event coordinates.

In [ ]:
import pandas as pd
import numpy as np
# Authenticate and Initialize Earth Engine
logging.info("Authenticating Google Earth Engine...")
auth.authenticate_user()
ee.Initialize(project='rational-terra-501919-n7') # Replace with your GCP project ID
logging.info("Google Earth Engine initialized.\n")

# Identify unique events based on pyroCb_id
# For each pyroCb_id, find a representative latitude and longitude.
# Prioritize valid coordinates over NaN. If all are NaN, then the representative will be NaN.

def get_representative_coords(group):
    # Try to find a row with non-null pixel_latitude and pixel_longitude
    valid_coords = group.dropna(subset=['pixel_latitude', 'pixel_longitude'])
    if not valid_coords.empty:
        # Return the first valid coordinate pair
        return valid_coords.iloc[0][['pixel_latitude', 'pixel_longitude']]
    else:
        # If all coordinates are NaN for this pyroCb_id, return NaNs
        return pd.Series({'pixel_latitude': np.nan, 'pixel_longitude': np.nan})

unique_events_for_processing = df_merged.groupby('pyroCb_id').apply(get_representative_coords).reset_index()

# Rename the columns for clarity in the loop
unique_events_for_processing.rename(columns={'pixel_latitude': 'reference_latitude', 'pixel_longitude': 'reference_longitude'}, inplace=True)

logging.info(f"Identified {len(unique_events_for_processing)} unique events for terrain data extraction.\n")
display(unique_events_for_processing.head())

/tmp/ipykernel_27592/321498480.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  unique_events_for_processing = df_merged.groupby('pyroCb_id').apply(get_representative_coords).reset_index()


,pyroCb_id,reference_latitude,reference_longitude
0,179,33.287442,-108.588483
1,180,33.202715,-111.082459
2,181,37.712003,-113.792203
3,189,41.473988,-122.333748
4,190,57.515594,-123.003883


### Processing Each Unique Event

For each unique event, the following steps will be performed:
1.  **Google Earth Engine Query:** Terrain features (Elevation, Slope, Aspect, TPI, TRI) will be calculated.
2.  **GeoTIFF Download:** The processed terrain data will be downloaded as a GeoTIFF file to your Google Drive.
3.  **Feature Extraction:** The terrain features at the exact `pixel_latitude` and `pixel_longitude` will be extracted from the downloaded GeoTIFF.

This process will be logged for tracking progress.

## Step 4: Core Terrain Processing Pipeline
This cell loops through your events and extracts Elevation, Slope, Aspect, TPI, and TRI using SRTM (low latitudes) or JAXA ALOS (high latitudes).

In [ ]:
all_extracted_terrain_data = []
target_directory = '/content/drive/MyDrive/Shared_data/tmp_files_can_be_cleanup_later'
os.makedirs(target_directory, exist_ok=True)
SRTM_LAT_LIMIT = 60

for index, row in unique_events_for_processing.iterrows():
    pid = row['pyroCb_id']
    lat, lon = row['reference_latitude'], row['reference_longitude']
    logging.info(f"\nProcessing event ID: {pid} (Lat: {lat}, Lon: {lon})")

    if pd.isna(lat) or pd.isna(lon):
        all_extracted_terrain_data.append({'pyroCb_id': pid, 'elevation': np.nan, 'slope': np.nan, 'aspect_sin': np.nan, 'aspect_cos': np.nan, 'tpi': np.nan, 'tri': np.nan})
        continue

    try:
        # Determine dataset: SRTM for <= 60 deg, JAXA for > 60 deg
        if abs(lat) <= SRTM_LAT_LIMIT:
            logging.info(f"Using SRTM (90m) for event {pid}")
            dataset = ee.Image('CGIAR/SRTM90_V4')
            elevation = dataset.select('elevation')
            scale = 90
        else:
            logging.info(f"Using JAXA ALOS (30m) for high-latitude event {pid}")
            dataset = ee.ImageCollection("JAXA/ALOS/AW3D30/V4_1")
            elevation = dataset.select('DSM').median()
            scale = 30

        terrain = ee.Terrain.products(elevation)
        slope = terrain.select('slope')
        aspect = terrain.select('aspect')
        aspectRad = aspect.multiply(math.pi / 180)
        aspectSin = aspectRad.sin().rename('aspect_sin')
        aspectCos = aspectRad.cos().rename('aspect_cos')
        tpi = elevation.subtract(elevation.focalMean(radius=300, units='meters')).rename('TPI')
        tri = elevation.reduceNeighborhood(reducer=ee.Reducer.mean(), kernel=ee.Kernel.square(radius=1, units='pixels')).subtract(elevation).abs().rename('TRI')

        stack = ee.Image.cat([elevation, slope, aspectSin, aspectCos, tpi, tri]).toFloat()
        region = ee.Geometry.Point([lon, lat]).buffer(500).bounds()
        url = stack.getDownloadURL({'scale': scale, 'crs': 'EPSG:4326', 'region': region, 'format': 'GEO_TIFF'})

        save_path = os.path.join(target_directory, f"terrain_{pid}.tif")
        resp = requests.get(url)
        with open(save_path, 'wb') as f: f.write(resp.content)

        with rasterio.open(save_path) as src:
            r, c = src.index(lon, lat)
            vals = src.read()[:, r, c]
            all_extracted_terrain_data.append({
                'pyroCb_id': pid, 'elevation': vals[0], 'slope': vals[1], 'aspect_sin': vals[2],
                'aspect_cos': vals[3], 'tpi': vals[4], 'tri': vals[5]
            })
            logging.info(f"Extracted: {all_extracted_terrain_data[-1]}")

    except Exception as e:
        logging.error(f"Error processing {pid}: {e}")
        all_extracted_terrain_data.append({'pyroCb_id': pid, 'elevation': np.nan, 'slope': np.nan, 'aspect_sin': np.nan, 'aspect_cos': np.nan, 'tpi': np.nan, 'tri': np.nan})

logging.info("Finished processing all events with JAXA fallback.")

### Merging Terrain Features with Original Data

Now, the extracted terrain features will be merged back into the original `df_merged` DataFrame. The merge will be based on `pyroCb_id`, effectively adding the terrain columns to all rows that share the same event ID.

## Step 5: Data Integration & Export
Merge the newly extracted terrain features back into your main dataset and save the final CSV.

In [ ]:
# Convert the list of extracted terrain data into a DataFrame
df_terrain_features = pd.DataFrame(all_extracted_terrain_data)

logging.info("Merging extracted terrain features with the original DataFrame...")
# Merge the terrain features back to the original DataFrame based on pyroCb_id
# This replaces any old terrain columns if they existed in the initial load
columns_to_keep = [col for col in df_merged.columns if col not in ['elevation', 'slope', 'aspect_sin', 'aspect_cos', 'tpi', 'tri']]
df_merged_clean = df_merged[columns_to_keep]

df_merged_final = pd.merge(
    df_merged_clean,
    df_terrain_features,
    on='pyroCb_id',
    how='left'
)

logging.info("Merge complete. Displaying head of the final DataFrame (Step 5):")
display(df_merged_final.head())

,pyroCb_id,time,pixel_latitude,pixel_longitude,latitude_vegetation,longitude_vegetation,cvh,cvl,tvh,tvl,...,cin_filled,injection_potential,PII,capped_flag,elevation,slope,aspect_sin,aspect_cos,tpi,tri
0,179,5/27/2021 6:00,33.287442,-108.588483,33.25,-108.5,0.888824,0.093579,3,16,...,0.0,-0.214816,-0.019456,False,3040.0,29.0,0.374607,0.927184,4.567567,1.0
1,179,5/27/2021 12:00,33.287442,-108.588483,33.25,-108.5,0.888824,0.093579,3,16,...,0.0,-0.405668,-0.215182,False,3040.0,29.0,0.374607,0.927184,4.567567,1.0
2,179,5/27/2021 18:00,33.287442,-108.588483,33.25,-108.5,0.888824,0.093579,3,16,...,0.0,-0.183975,-0.223115,False,3040.0,29.0,0.374607,0.927184,4.567567,1.0
3,179,5/28/2021 0:00,33.287442,-108.588483,33.25,-108.5,0.888824,0.093579,3,16,...,0.0,-0.199941,-0.204530,False,3040.0,29.0,0.374607,0.927184,4.567567,1.0
4,179,5/28/2021 6:00,33.287442,-108.588483,33.25,-108.5,0.888824,0.093579,3,16,...,0.0,-0.449490,-0.247899,False,3040.0,29.0,0.374607,0.927184,4.567567,1.0


### Saving the Final DataFrame

Finally, the DataFrame with all the original data and the newly added terrain features will be saved to a new CSV file in your Google Drive.

In [ ]:
# Define the path for the new merged CSV file
output_csv_path = '/content/drive/MyDrive/Shared_data/merged_with_terrain_Terrain_V1.csv'

logging.info(f"Saving the final consolidated DataFrame to {output_csv_path}...")
df_merged_final.to_csv(output_csv_path, index=False)
logging.info(f"Final file saved. This file now contains SRTM data for low latitudes and JAXA data for high latitudes.")

## Step 6: Final Verification
Reload the saved file and check specific high-latitude events to confirm the JAXA data was integrated correctly.

In [ ]:
# Load the newly saved file
df_check = pd.read_csv(output_csv_path)

# Check the specific problematic IDs
check_ids = [216.0, 258.0, 260.0]
verification = df_check[df_check['pyroCb_id'].isin(check_ids)][['pyroCb_id', 'elevation', 'slope', 'aspect_sin', 'aspect_cos', 'tpi', 'tri']].drop_duplicates()

print("Verification of high-latitude events (JAXA Results):")
display(verification)

# Summary of missing values across the whole dataset
print("\nMissing values summary for terrain columns:")
print(df_check[['elevation', 'slope', 'tpi', 'tri']].isna().sum())

Verification of high-latitude events (JAXA Results):


,pyroCb_id,elevation,slope,aspect_sin,aspect_cos,tpi,tri
137,216,967.0,0.0,-0.798636,0.601815,1.245902,1.000000
180,258,384.0,0.0,0.984808,0.173648,-1.003279,0.888889
203,260,621.0,0.0,-0.990268,-0.139173,2.842623,3.333333



Missing values summary for terrain columns:
elevation    0
slope        0
tpi          0
tri          0
dtype: int64


### Standalone Test: Using ASTER GDEM for high-latitude events

This section demonstrates how to use the `NASA/ASTER_VNIR_V2` dataset for events that fall outside the SRTM data coverage (e.g., higher latitudes). This is a standalone test and does not affect the `df_merged_final` DataFrame created earlier.

In [ ]:
all_extracted_terrain_data_aster = []

# Specific pyroCb_ids to re-process with ASTER
problematic_pyroCb_ids = [216.0, 258.0, 260.0]

# Filter unique events to only include the problematic ones for this test
# We'll re-fetch the unique_events_for_processing or define a similar structure for this test

# Re-create a minimal unique_events_for_processing for this standalone test
# to avoid dependency on previous cells if this is run independently
test_events_df = df_merged[['pyroCb_id', 'pixel_latitude', 'pixel_longitude']].drop_duplicates().reset_index(drop=True)
events_to_reprocess_aster = test_events_df[test_events_df['pyroCb_id'].isin(problematic_pyroCb_ids)]
events_to_reprocess_aster.rename(columns={'pixel_latitude': 'reference_latitude', 'pixel_longitude': 'reference_longitude'}, inplace=True)

logging.info(f"\nStarting standalone re-processing for problematic events using NASA/ASTER_VNIR_V2.")

for index, row in events_to_reprocess_aster.iterrows():
    current_pyroCb_id = row['pyroCb_id']
    reference_lat = row['reference_latitude']
    reference_lon = row['reference_longitude']

    logging.info(f"\nProcessing event ID: {current_pyroCb_id} (Lat: {reference_lat}, Lon: {reference_lon}) with ASTER GDEM")

    # Skip if coordinates are NaN (should have been handled by previous filtering, but as a safeguard)
    if pd.isna(reference_lat) or pd.isna(reference_lon):
        logging.warning(f"Skipping ASTER processing for event ID {current_pyroCb_id} (Lat: {reference_lat}, Lon: {reference_lon}) due to NaN coordinates. Recording NaNs.")
        all_extracted_terrain_data_aster.append({
            'pyroCb_id': current_pyroCb_id,
            'elevation': np.nan,
            'slope': np.nan,
            'aspect_sin': np.nan,
            'aspect_cos': np.nan,
            'tpi': np.nan,
            'tri': np.nan
        })
        continue

    try:
        # 1. Google Earth Engine Terrain Processing with ASTER (NASA/ASTER_VNIR_V2)
        # ASTER GDEM v2 has an 'elevation' band directly
        dataset_aster = ee.Image('NASA/ASTER_VNIR_V2')
        elevation_aster = dataset_aster.select('elevation')

        terrain_aster = ee.Terrain.products(elevation_aster)
        slope_aster = terrain_aster.select('slope')
        aspect_aster = terrain_aster.select('aspect')

        aspectRad_aster = aspect_aster.multiply(math.pi / 180)
        aspectSin_aster = aspectRad_aster.sin().rename('aspect_sin')
        aspectCos_aster = aspectRad_aster.cos().rename('aspect_cos')

        localMean_aster = elevation_aster.focalMean(radius=300, units='meters')
        tpi_aster = elevation_aster.subtract(localMean_aster).rename('TPI')

        squareKernel_aster = ee.Kernel.square(radius=1, units='pixels')
        tri_aster = elevation_aster.reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=squareKernel_aster
        ).subtract(elevation_aster).abs().rename('TRI')

        pyroCB_terrain_stack_aster = ee.Image.cat([
            elevation_aster,
            slope_aster,
            aspectSin_aster,
            aspectCos_aster,
            tpi_aster,
            tri_aster
        ]).toFloat()

        # Define Export Region around the reference point
        buffer_deg = 0.1
        exportRegion_aster = ee.Geometry.Rectangle([
            reference_lon - buffer_deg,
            reference_lat - buffer_deg,
            reference_lon + buffer_deg,
            reference_lat + buffer_deg
        ])

        url_aster = pyroCB_terrain_stack_aster.getDownloadURL({
            'scale': 30, # ASTER is ~30m resolution
            'crs': 'EPSG:4326',
            'region': exportRegion_aster,
            'format': 'GEO_TIFF'
        })
        logging.info(f"ASTER GEE processing complete. Download URL generated.")

        # 2. Download the GeoTIFF file
        filename_aster = f"terrain_data_ASTER_{current_pyroCb_id}_{reference_lat}_{reference_lon}.tif"
        save_path_aster = os.path.join(target_directory, filename_aster)

        logging.info(f"Downloading ASTER GeoTIFF from: {url_aster}")
        response_aster = requests.get(url_aster, stream=True)
        response_aster.raise_for_status()

        with open(save_path_aster, 'wb') as f:
            for chunk in response_aster.iter_content(chunk_size=8192):
                f.write(chunk)
        logging.info(f"ASTER GeoTIFF downloaded successfully to {save_path_aster}")

        # 3. Extract terrain features from the downloaded GeoTIFF
        extracted_features_aster = {
            'pyroCb_id': current_pyroCb_id
        }

        with rasterio.open(save_path_aster) as src_aster:
            try:
                row_idx_aster, col_idx_aster = src_aster.index(reference_lon, reference_lat)
                pixel_values_aster = src_aster.read()[:, row_idx_aster, col_idx_aster]

                extracted_features_aster.update({
                    'elevation': pixel_values_aster[0],
                    'slope': pixel_values_aster[1],
                    'aspect_sin': pixel_values_aster[2],
                    'aspect_cos': pixel_values_aster[3],
                    'tpi': pixel_values_aster[4],
                    'tri': pixel_values_aster[5]
                })
                logging.info(f"Extracted ASTER features: {extracted_features_aster}")
            except IndexError:
                logging.warning(f"Coordinate {reference_lon}, {reference_lat} is outside the bounds of the downloaded ASTER GeoTIFF. Recording NaNs.")
                extracted_features_aster.update({
                    'elevation': np.nan,
                    'slope': np.nan,
                    'aspect_sin': np.nan,
                    'aspect_cos': np.nan,
                    'tpi': np.nan,
                    'tri': np.nan
                })
        all_extracted_terrain_data_aster.append(extracted_features_aster)

    except Exception as e:
        logging.error(f"Error processing event ID {current_pyroCb_id} (Lat: {reference_lat}, Lon: {reference_lon}) with ASTER: {e}. Recording NaNs.")
        all_extracted_terrain_data_aster.append({
            'pyroCb_id': current_pyroCb_id,
            'elevation': np.nan,
            'slope': np.nan,
            'aspect_sin': np.nan,
            'aspect_cos': np.nan,
            'tpi': np.nan,
            'tri': np.nan
        })

logging.info("Finished processing problematic events with ASTER GDEM.")
df_terrain_features_aster_test = pd.DataFrame(all_extracted_terrain_data_aster)
display(df_terrain_features_aster_test)

In [ ]:
import ee
import pandas as pd
import numpy as np
import math
import requests
import os
import rasterio
import logging

# 1. Initialize variables for JAXA V4_1 test
problematic_pyroCb_ids = [216.0, 258.0, 260.0]
target_directory = '/content/drive/MyDrive/Shared_data/tmp_files_can_be_cleanup_later'
os.makedirs(target_directory, exist_ok=True)
all_extracted_terrain_jaxa = []

# Filter unique events
events_to_reprocess_jaxa = df_merged[df_merged['pyroCb_id'].isin(problematic_pyroCb_ids)][['pyroCb_id', 'pixel_latitude', 'pixel_longitude']].drop_duplicates()

logging.info("Starting standalone re-processing using JAXA/ALOS/AW3D30/V4_1")

for index, row in events_to_reprocess_jaxa.iterrows():
    pid = row['pyroCb_id']
    lat, lon = row['pixel_latitude'], row['pixel_longitude']

    try:
        # Load the latest ALOS collection and create a median composite (or use a mosaic)
        # V4_1 is the current version
        jaxa_col = ee.ImageCollection("JAXA/ALOS/AW3D30/V4_1")
        elevation = jaxa_col.select('DSM').median()

        # Terrain products (Slope, Aspect)
        terrain = ee.Terrain.products(elevation)
        slope = terrain.select('slope')
        aspect = terrain.select('aspect')

        # Calculate Sin/Cos for Aspect
        aspectRad = aspect.multiply(math.pi / 180)
        aspectSin = aspectRad.sin().rename('aspect_sin')
        aspectCos = aspectRad.cos().rename('aspect_cos')

        # TPI & TRI logic
        localMean = elevation.focalMean(radius=300, units='meters')
        tpi = elevation.subtract(localMean).rename('TPI')
        tri = elevation.reduceNeighborhood(reducer=ee.Reducer.mean(), kernel=ee.Kernel.square(radius=1, units='pixels')).subtract(elevation).abs().rename('TRI')

        # Stack
        stack = ee.Image.cat([elevation, slope, aspectSin, aspectCos, tpi, tri]).toFloat()

        # Export Region buffer
        region = ee.Geometry.Point([lon, lat]).buffer(500).bounds()

        url = stack.getDownloadURL({
            'scale': 30,
            'crs': 'EPSG:4326',
            'region': region,
            'format': 'GEO_TIFF'
        })

        save_path = os.path.join(target_directory, f"jaxa_v4_test_{pid}.tif")
        resp = requests.get(url)
        with open(save_path, 'wb') as f: f.write(resp.content)

        with rasterio.open(save_path) as src:
            r, c = src.index(lon, lat)
            vals = src.read()[:, r, c]
            all_extracted_terrain_jaxa.append({
                'pyroCb_id': pid,
                'elevation': vals[0],
                'slope': vals[1],
                'aspect_sin': vals[2],
                'aspect_cos': vals[3],
                'tpi': vals[4],
                'tri': vals[5]
            })
            logging.info(f"Event {pid} processed successfully using JAXA V4.1.")

    except Exception as e:
        logging.error(f"Failed event {pid}: {e}")

df_jaxa_results = pd.DataFrame(all_extracted_terrain_jaxa)
display(df_jaxa_results)

### Understanding the Terrain Features

Each column in the extracted `df` represents a specific physical characteristic of the landscape that influences fire behavior:

*   **Elevation (meters):** Fire generally travels faster uphill. Furthermore, higher elevations often have cooler temperatures and different vegetation types compared to valleys.
*   **Slope (degrees):** This is one of the most critical factors. A steeper slope allows the flames to 'pre-heat' the fuel (trees/brush) above them more effectively, leading to rapid uphill fire spread.
*   **Aspect (Sin/Cos):** This describes the direction the slope faces.
    *   **South-facing slopes** (in the Northern Hemisphere) receive more direct sunlight, making them drier and more prone to ignition.
    *   We use **Sine** and **Cosine** to represent direction mathematically so the model understands that 359° is right next to 1°.
*   **TPI (Topographic Position Index):** This tells us if a point is a **ridge** (positive value) or a **valley** (negative value) relative to its surroundings. Valleys can act as chimneys that funnel wind and heat.
*   **TRI (Terrain Ruggedness Index):** This measures how 'rough' or 'broken' the ground is. Highly rugged terrain can create unpredictable local wind eddies, making fire behavior harder to forecast.

### Google Earth Engine Terrain Processing Pipeline
This cell replicates your JavaScript logic to compute a multi-band terrain image containing Elevation, Slope, Aspect (Sin/Cos), TPI, and TRI.